In [1]:
# QUESTION 1
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50

IMG_SIZE = (224, 224)
NUM_CLASSES = 4

# Data augmentation
augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2)
])

# Pre-trained CNN
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze the initial layers
base_model.trainable = False

# Build model
inputs = layers.Input(shape=(224, 224, 3))
x = augmentation(inputs)

x = tf.keras.applications.resnet50.preprocess_input(x)
x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# Train
# model.fit(train_dataset, validation_data=val_dataset, epochs=10)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 224, 224)  │          0 │ sequential[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 224, 224)  │          0 │ sequential[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 224, 224)  │          0 │ sequential[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 224, 224,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 224, 224,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add[0][0]         │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    524,544 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 4)         │      1,028 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,113,284 (91.98 MB)

 Trainable params: 525,572 (2.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [2]:
# Unfreeze the last part of ResNet50
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [3]:
#QUESTION 2
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import DenseNet121

IMG_SIZE = 256
NUM_CLASSES = 8

# DenseNet encoder
base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

# Input
inputs = layers.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Feature extraction
x = base_model(inputs)

# Decoder
x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)

x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)

x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)

x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)

x = layers.UpSampling2D((2, 2))(x)

# Pixel-wise classification
outputs = layers.Conv2D(
    NUM_CLASSES,
    kernel_size=1,
    activation="softmax",
    padding="same"
)(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 8, 8, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 8, 8, 256)      │     2,359,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d (UpSampling2D)    │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 128)    │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_1 (UpSampling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 64)     │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_2 (UpSampling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 32)     │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_3 (UpSampling2D)  │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 128, 128, 8)    │           264 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,784,616 (37.33 MB)

 Trainable params: 2,747,112 (10.48 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [4]:
#QUESTION 3
import tensorflow as tf
from tensorflow.keras import layers

VOCAB_SIZE = 10000
EMBEDDING_DIM = 256
LATENT_DIM = 256
MAX_LENGTH = 30

# -------------------------
# Encoder
# -------------------------

encoder_inputs = layers.Input(
    shape=(MAX_LENGTH,),
    name="encoder_input"
)

encoder_embedding = layers.Embedding(
    VOCAB_SIZE,
    EMBEDDING_DIM
)(encoder_inputs)

encoder = layers.Bidirectional(
    layers.LSTM(
        LATENT_DIM,
        return_sequences=True,
        return_state=True
    )
)

encoder_outputs, forward_h, forward_c, backward_h, backward_c = \
    encoder(encoder_embedding)

# Combine forward and backward states
state_h = layers.Concatenate()(
    [forward_h, backward_h]
)

state_c = layers.Concatenate()(
    [forward_c, backward_c]
)

# -------------------------
# Decoder
# -------------------------

decoder_inputs = layers.Input(
    shape=(MAX_LENGTH,),
    name="decoder_input"
)

decoder_embedding = layers.Embedding(
    VOCAB_SIZE,
    EMBEDDING_DIM
)(decoder_inputs)

decoder_lstm = layers.LSTM(
    LATENT_DIM * 2,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=[state_h, state_c]
)

decoder_outputs = layers.Dense(
    VOCAB_SIZE,
    activation="softmax"
)(decoder_outputs)

# Complete model
model = tf.keras.Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 30, 256)   │  2,560,000 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ [(None, 30, 512), │  1,050,624 │ embedding[0][0]   │
│ (Bidirectional)     │ (None, 256),      │            │                   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 30, 256)   │  2,560,000 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 512)       │          0 │ bidirectional[0]… │
│ (Concatenate)       │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 512)       │          0 │ bidirectional[0]… │
│ (Concatenate)       │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 30, 512), │  1,574,912 │ embedding_1[0][0… │
│                     │ (None, 512),      │            │ concatenate[0][0… │
│                     │ (None, 512)]      │            │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 30, 10000) │  5,130,000 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 12,875,536 (49.12 MB)

 Trainable params: 12,875,536 (49.12 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
#QUESTION 4
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

# -------------------------------------------------
# 1. Create sample historical stock-price data
# -------------------------------------------------

np.random.seed(42)
tf.random.set_seed(42)

days = 1000

prices = [100]

for i in range(1, days):
    change = np.random.normal(0, 1)
    new_price = prices[-1] + change
    new_price = max(new_price, 1)
    prices.append(new_price)

prices = np.array(prices).reshape(-1, 1)

print("Total data points:", len(prices))


# -------------------------------------------------
# 2. Normalize the data
# -------------------------------------------------

scaler = MinMaxScaler()

scaled_data = scaler.fit_transform(prices)


# -------------------------------------------------
# 3. Create 30-day sequences
# -------------------------------------------------

TIME_STEPS = 30

X = []
y = []

for i in range(TIME_STEPS, len(scaled_data)):

    X.append(
        scaled_data[i - TIME_STEPS:i]
    )

    y.append(
        scaled_data[i]
    )

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)


# -------------------------------------------------
# 4. Split into training and testing data
# -------------------------------------------------

train_size = int(len(X) * 0.8)

X_train = X[:train_size]
y_train = y[:train_size]

X_test = X[train_size:]
y_test = y[train_size:]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


# -------------------------------------------------
# 5. Build LSTM model
# -------------------------------------------------

model = Sequential()

model.add(
    LSTM(
        64,
        return_sequences=True,
        input_shape=(TIME_STEPS, 1)
    )
)

model.add(Dropout(0.2))

model.add(
    LSTM(32)
)

model.add(Dropout(0.2))

model.add(
    Dense(16, activation="relu")
)

model.add(
    Dense(1)
)


# -------------------------------------------------
# 6. Compile model
# -------------------------------------------------

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mean_squared_error",
    metrics=["mae"]
)


# -------------------------------------------------
# 7. Display model
# -------------------------------------------------

model.summary()


# -------------------------------------------------
# 8. Train model
# -------------------------------------------------

history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test),
    shuffle=False
)


# -------------------------------------------------
# 9. Evaluate model
# -------------------------------------------------

loss, mae = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\nTest Loss:", loss)
print("Test MAE:", mae)


# -------------------------------------------------
# 10. Predict test data
# -------------------------------------------------

predictions = model.predict(
    X_test,
    verbose=0
)


# Convert predictions back to actual prices
predicted_prices = scaler.inverse_transform(
    predictions
)

actual_prices = scaler.inverse_transform(
    y_test
)


# -------------------------------------------------
# 11. Display predictions
# -------------------------------------------------

print("\nSample Predictions:")

for i in range(10):

    print(
        "Actual:",
        round(actual_prices[i][0], 2),
        "| Predicted:",
        round(predicted_prices[i][0], 2)
    )


# -------------------------------------------------
# 12. Predict next day's price
# -------------------------------------------------

last_30_days = scaled_data[-TIME_STEPS:]

last_30_days = last_30_days.reshape(
    1,
    TIME_STEPS,
    1
)

next_prediction = model.predict(
    last_30_days,
    verbose=0
)

next_price = scaler.inverse_transform(
    next_prediction
)[0][0]

current_price = prices[-1][0]

print("\nCurrent Price:",
      round(current_price, 2))

print("Predicted Next Price:",
      round(next_price, 2))


# -------------------------------------------------
# 13. Predict UP / DOWN trend
# -------------------------------------------------

if next_price > current_price:

    print("Predicted Trend: UP")

else:

    print("Predicted Trend: DOWN")

Total data points: 1000
X shape: (970, 30, 1)
y shape: (970, 1)
Training samples: 776
Testing samples: 194


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 30, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,857 (116.63 KB)

 Trainable params: 29,857 (116.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - loss: 0.0150 - mae: 0.1010 - val_loss: 0.1607 - val_mae: 0.3586
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step - loss: 0.0221 - mae: 0.1252 - val_loss: 0.0578 - val_mae: 0.2000
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - loss: 0.0188 - mae: 0.1150 - val_loss: 0.0315 - val_mae: 0.1468
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - loss: 0.0127 - mae: 0.0911 - val_loss: 0.0111 - val_mae: 0.0871
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.0069 - mae: 0.0649 - val_loss: 0.0042 - val_mae: 0.0503
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.0041 - mae: 0.0493 - val_loss: 0.0083 - val_mae: 0.0763
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.0036 - mae: 0.0462 - val_loss: 0.0138 - val_mae: 0.1023
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0042 - mae: 0.0507 - val_loss: 0.0097 - val_mae: 0.0839
Epoch 9/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.004

In [7]:
#QUESTION 5
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50

IMG_SIZE = 224
FEATURE_SIZE = 2048
LSTM_UNITS = 512
VOCAB_SIZE = 10000
MAX_LENGTH = 30

# =====================================================
# 1. CNN FEATURE EXTRACTOR
# =====================================================

cnn = ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze CNN
cnn.trainable = False

# Example:
# frame -> CNN -> 2048-dimensional feature vector
frame_input = layers.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)

frame_feature = cnn(frame_input)

feature_extractor = Model(
    frame_input,
    frame_feature
)

feature_extractor.summary()


Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 2048)           │    23,587,712 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,587,712 (89.98 MB)